# Analysis of Short Term Rentals on Airbnb in Lombardy

## Introduction

This project aims to identify the key drivers of demand for short term rentals in Lombardy, specifically in the cities of Milan and Bergamo. <br>
For each city individually, we aim to estimate the contribution that factors such as _price_, _amenities_, _neighborhood_ and others bring to the overall demand signal for a listing.  

## Data Exploration

### Data Loading

In [ ]:
import pandas as pd

df_mi = pd.read_csv("../../data/listings_milan.csv")
df_bg = pd.read_csv("../../data/listings_bergamo.csv")

df_mi.head()

In [ ]:
df_mi.info()

In [ ]:
df_bg.info()

In [ ]:
## Adding an unambiguous first column to avoid collisions later
df_mi["unique_id"] = df_mi["id"].astype(str) + "_mi"
df_mi["city"] = "MI"
df_bg["unique_id"] = df_bg["id"].astype(str) + "_bg"
df_bg["city"] = "BG"


n_rows_mi = df_mi.shape[0]
n_rows_bg = df_bg.shape[0]

column_order = ["unique_id", "city"] + [col for col in df_mi.columns if col not in ("unique_id", "city")]
df_mi = df_mi[column_order]
df_bg = df_bg[column_order]

# Merging the two DFs
df = pd.concat([df_mi, df_bg], axis=0, ignore_index=True)
assert n_rows_bg + n_rows_mi == df.shape[0]

### for our specific use case, we select only a subset of the dataset's columns

In [ ]:

necessary_cols = [
    "unique_id",
    "city",
    "name",
    "description",
    "host_id",
    "host_since",
    "host_response_rate",
    "host_response_time",
    "minimum_nights",
    "maximum_nights",
    "estimated_occupancy_l365d",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "first_review",
    "last_review",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "accommodates",
    "bedrooms",
    "host_is_superhost",
    "host_has_profile_pic",
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
    "amenities",
    "price"
    ]

fill_values = {
    "name": "",
    "description": "",
    "host_id": 0,
    "host_since": "2000-01-01",              # Fixed: Passed as a clean string
    "host_response_rate": "0%",               # Fixed: Corrected typo from hotel -> host
    "host_response_time": "unknown",
    "minimum_nights": 1,
    "maximum_nights": 365,
    "estimated_occupancy_l365d": 0,
    "number_of_reviews_l30d": 0,
    "number_of_reviews_ly": 0,
    "first_review": "2000-01-01",             # Fixed: Passed as a clean string
    "last_review": "2000-01-01",              # Fixed: Passed as a clean string
    "availability_30": 0,
    "availability_60": 0,
    "availability_90": 0,
    "availability_365": 0,
    "accommodates": 1,
    "bedrooms": 1,
    "host_is_superhost": "f",
    "host_has_profile_pic": "f",
    "neighbourhood_cleansed": "unknown",
    "property_type": "unknown",
    "room_type": "unknown",
    "amenities": "[]"                       
}  # WE DO NOT FILL PRICE

df = df[necessary_cols]
df.head()

In [ ]:
## Running check on NULL entries
import sys 
sys.path.append("../")
from utils.null_analyzer import NullAnalyzer

null_analyzer = NullAnalyzer(threshold=5, stratify_col="city")

null_analyzer.check_nulls_globally(df=df)
null_analyzer.check_nulls_strat(df=df, delta_threshold=5)

In [ ]:
df["has_host_been_contacted"] = df["host_response_rate"].isna()
df["host_never_had_reviews"] = df["first_review"].isna()
df = df.fillna(value=fill_values).dropna()

In [ ]:
snapshot_date = pd.to_datetime(df["last_scraped"].iloc[0] if "last_scraped" in df.columns else "2025-09-27")
df["host_age_days"] = (snapshot_date - pd.to_datetime(df["host_since"], errors="coerce")).dt.days

n_before = df.shape[0]
df = df[df["host_age_days"] >= 365]
n_after = df.shape[0]
print(f"Filtered out {n_before - n_after} listings with host age < 1 year")
print(f"Retained {n_after} listings with host age >= 1 year")

occupancy_windows = [30, 60, 90, 365]
for window in occupancy_windows:
    df[f"occupancy_{window}"] = 1 - (df[f"availability_{window}"] / window)

### Analysis of Occupancy_90 variable

In [ ]:
## VERIFYING WHETHER MILAN AND BERGAMO TARGET VARIABLES ARE DISTRIBUTED DIFFERENTLY
from utils.target_var_inspection import plot_kde_by_city, generate_violin_plot_by_city, plot_categorical_barchart

plot_kde_by_city(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

In [ ]:
generate_violin_plot_by_city(df, target_column="occupancy_90", city_column="city", rng=(0,1))

The occupancy variable has strong peaks in both Milan around the 0.33, 0.66 and 1.0 marks, signalling that there are likely listings where the owners blocked the whole timeperiod. <br>
We must therefore implement some logics to separate the true fully booked listings from the blocked ones. <br>

In [ ]:
plot_kde_by_city(df, target_column="minimum_nights", city_column="city", rng=(0, 50))

In [ ]:
plot_kde_by_city(df, target_column="maximum_nights", city_column="city", rng=(0, 2000))

We clearly see that the minimum nights distribution is bimodal, with peaks around 1 day for the actual short term renters, and around 30 for the long term ones. <br>
For the purpose of this analysis, we can restrict our focus to the short term renters.

In [ ]:
# Restricting the scope to short term rentals
n_before = df.shape[0]
df = df[df["minimum_nights"] < 15]
n_after = df.shape[0]
print(f"Filtered out {n_before - n_after} listings with minimum_nights >= 15")
print(f"Retained {n_after} listings with minimum_nights < 15")

In [ ]:
plot_categorical_barchart(df, category_col="host_response_time")

In [ ]:
df["host_response_rate"] = df["host_response_rate"].astype(str).str.rstrip("%")

df["host_response_rate"] = pd.to_numeric(
    df["host_response_rate"], errors="coerce"
).astype("Int64")

In [ ]:
plot_kde_by_city(df[~df["has_host_been_contacted"]], target_column="host_response_rate", rng=(0, 100))

In [ ]:
df["is_host_inactive"] = (df["host_response_rate"] < 90) | (df["host_response_time"] != "within an hour")

In [ ]:
## Preparing data
df["last_review_dt"] = pd.to_datetime(df["last_review"], errors="coerce")
df["first_review_dt"] = pd.to_datetime(df["first_review"], errors="coerce")

df["reviews_per_year"] = pd.to_numeric(df["number_of_reviews_ly"], errors="coerce")
df["availability_365"] = pd.to_numeric(df["availability_365"], errors="coerce")

df["is_host_inactive"] = df["is_host_inactive"].astype(bool)
df["host_never_had_reviews"] = df["host_never_had_reviews"].astype(bool)

# Occupancy values not defined are set to 0
df["occupancy_90"] = pd.to_numeric(df["occupancy_90"], errors="coerce")
df["occupancy_60"] = pd.to_numeric(df["occupancy_60"], errors="coerce")
df["occupancy_30"] = pd.to_numeric(df["occupancy_30"], errors="coerce")


In [ ]:
import numpy as np

high_occupancy_90 = df["occupancy_90"] > 0.90
high_occupancy_60 = df["occupancy_60"] > 0.95
high_occupancy_30 = df["occupancy_30"] == 1.0


block_conditions = [
    high_occupancy_90,
    high_occupancy_60,
    high_occupancy_30
]

review_floor_choices = [6, 4, 2]

df["required_review_floor"] = np.select(block_conditions, review_floor_choices, default=0)

cond_review_stale = (snapshot_date - df["last_review_dt"]).dt.days > 30    # hasn't been reviews in a while
is_older_than_1_year = (snapshot_date - df["first_review_dt"]).dt.days > 365 # has been listed for more than a year

n_before = df.shape[0]
df = df[is_older_than_1_year]    # retaining only listings that have hosts active for more than 1 year
n_after = df.shape[0]
print(f"Filtered out {n_before - n_after} listings with host age < 1")
print(f"Retained {n_after} listings with host age >= 1")

cond_low_momentum = (df["reviews_per_year"] < df["required_review_floor"]) 

cond_calendar_dead = df["availability_365"] == 0

cond_host_dead = (df["is_host_inactive"] | df["host_never_had_reviews"]) == True

any_trigger_tripped = (
    cond_review_stale | 
    cond_low_momentum | 
    cond_calendar_dead | 
    cond_host_dead
)

df["block_90"] = high_occupancy_90 & any_trigger_tripped
df["block_60"] = high_occupancy_60 & any_trigger_tripped & ~high_occupancy_90
df["block_30"] = high_occupancy_30 & any_trigger_tripped & ~high_occupancy_60 & ~high_occupancy_90

df["is_blocked"] = df["block_90"] | df["block_60"] | df["block_30"]


# --- Step 5: Updated Verification Log ---
print("=== DYNAMIC BLOCK FILTER DISTRIBUTION ===")
print(f"Total Portfolio Size:          {len(df)}")
print(f"Total Blocked Units Flagged:   {df['is_blocked'].sum()}")
print(f"  - 90-Day Structural Blocks:  {df['block_90'].sum()}")
print(f"  - 60-Day Mid-Horizon Blocks: {df['block_60'].sum()}")
print(f"  - 30-Day Short-Horizon Blocks:{df['block_30'].sum()}")
print("-----------------------------------------")
print("=== BEHAVIORAL WARNING BREAKDOWN (MATURE HOSTS ONLY) ===")
has_any_occupancy_block = (high_occupancy_90 | high_occupancy_60 | high_occupancy_30)
print(f"  - Due to Stale Reviews:      {(has_any_occupancy_block & cond_review_stale).sum()}")
print(f"  - Due to Low Velocity (LTM): {(has_any_occupancy_block & cond_low_momentum).sum()}")
print(f"  - Due to 365-Day Lockout:    {(has_any_occupancy_block & cond_calendar_dead).sum()}")
print(f"  - Due to Host Inactivity:    {(has_any_occupancy_block & cond_host_dead).sum()}")

In [ ]:
n_before = df.shape[0]
df = df[~df["is_blocked"]]
n_after = df.shape[0]
print(f"Filtered out {n_before - n_after} listings flagged as blocked")
print(f"Retained {n_after} listings not flagged as blocked")

In [ ]:
plot_kde_by_city(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

In [ ]:
generate_violin_plot_by_city(df, target_column="occupancy_90", city_column="city", rng=(0,1))

In [ ]:
cond_listing_dead = ((snapshot_date - df["last_review_dt"]).dt.days > 60) & (df["occupancy_90"] == 0)
cond_listing_not_managed = (df["is_host_inactive"] & (df["occupancy_90"] == 0))
n_before = df.shape[0]
df = df[~(cond_listing_dead | cond_listing_not_managed)]
n_after = df.shape[0]
print(f"Filtered out {n_before - n_after} listings flagged as potentially dead")
print(f"Retained {n_after} listings not flagged as potentially dead")

In [ ]:
plot_kde_by_city(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

In [ ]:
from utils.target_var_inspection import plot_log_distribution_by_city

plot_log_distribution_by_city(df, variable_col="occupancy_90", city_col="city")